<a href="https://colab.research.google.com/github/Shineii86/LeechBot/blob/main/notebooks/LeechBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
<img src="https://capsule-render.vercel.app/api?type=waving&height=200&color=gradient&text=LeechBot&fontAlignY=30&fontSize=80&desc=Advanced%20Telegram%20File%20Transloader&descSize=20" />

**A powerful Pyrogram-based bot to transfer files to Telegram & Google Drive**

![Version](https://img.shields.io/badge/Version-3.0.2-8B5CF6?style=for-the-badge)
![Python](https://img.shields.io/badge/Python-3.10+-3776AB?style=for-the-badge&logo=python&logoColor=white)
![License](https://img.shields.io/badge/License-MIT-06B6D4?style=for-the-badge)

---

### ✨ Features

| 📥 Download From | 📤 Upload To | 🛠️ Tools |
|:---:|:---:|:---:|
| YouTube, Facebook, Instagram | Telegram | Video Converter (GPU) |
| Google Drive, Mega, Terabox | Google Drive | Archive Handler |
| Pixeldrain, Mediafire, Direct | Directory Leech | Smart Splitting |
| 2000+ sites via yt-dlp | Batch Photos | Download Queue |

---

### 🚀 Quick Start

1. **Fill credentials** in Cell 3 (or use Colab Secrets)
2. Click **Runtime → Run all** or press **Ctrl+F9**
3. Bot starts automatically — send `/start` on Telegram

</div>

In [ ]:
# @title ♻️ Google Drive Setup
#@markdown <div align="center">
#@markdown <img src="https://user-images.githubusercontent.com/125879861/255377947-6ac19c35-dbbd-4a9b-bc0e-c603de81c533.png" height="60">
#@markdown <h4>Google Drive Integration</h4>
#@markdown </div>

ACTION = "Mount Drive" # @param ["Mount Drive", "Unmount Drive", "Generate Token", "Skip"]
MOUNT_PATH = "/content/drive" # @param {type:"string"}
TOKEN_PATH = "/content/token.pickle" # @param {type:"string"}

import os, time, pickle
from IPython.display import display, Markdown, clear_output

def log(emoji, msg, color="cyan"):
    display(Markdown(f"<font color={color}>**{emoji} {msg}**</font>"))

if ACTION == "Skip":
    log("⏭️", "Skipped — Google Drive not needed", "gray")
else:
    from google.colab import auth, drive
    import google.auth
    from google.auth.transport.requests import Request

    if ACTION == "Mount Drive":
        for attempt in range(3):
            try:
                log("🔗", f"Mounting to {MOUNT_PATH}... (attempt {attempt+1}/3)")
                drive.mount(MOUNT_PATH, force_remount=True)
                log("✅", "Drive mounted!", "green")
                break
            except Exception as e:
                log("⚠️", f"Attempt {attempt+1} failed: {e}", "orange")
                time.sleep(2)

    elif ACTION == "Unmount Drive":
        try:
            drive.flush_and_unmount()
            log("🔓", "Drive unmounted", "green")
        except Exception as e:
            log("⚠️", f"Unmount: {e}", "orange")

    elif ACTION == "Generate Token":
        try:
            auth.authenticate_user()
            creds, _ = google.auth.default()
            if creds.expired and creds.refresh_token:
                creds.refresh(Request())
            with open(TOKEN_PATH, 'wb') as f:
                pickle.dump(creds, f)
            os.chmod(TOKEN_PATH, 0o600)
            log("✅", f"Token saved to {TOKEN_PATH}", "green")
        except Exception as e:
            log("❌", f"Token error: {e}", "red")

    if ACTION == "Mount Drive":
        try:
            auth.authenticate_user()
            creds, _ = google.auth.default()
            if creds.expired and creds.refresh_token:
                creds.refresh(Request())
            with open(TOKEN_PATH, 'wb') as f:
                pickle.dump(creds, f)
            os.chmod(TOKEN_PATH, 0o600)
            log("🔑", "GDrive token generated", "green")
        except Exception as e:
            log("⚠️", f"Token gen skipped: {e}", "orange")

log("💡", "Tip: Store credentials in Colab Secrets for auto-fill", "gray")

In [ ]:
# @title 🚀 Deploy LeechBot
#@markdown <div align="center">
#@markdown <img src="https://user-images.githubusercontent.com/125879861/255391401-371f3a64-732d-4954-ac0f-4f093a6605e1.png" width="500">
#@markdown </div>

#@markdown ---
#@markdown ## 🔐 Credentials
#@markdown > Fill manually or set in **🔑 Secrets** (left panel) with these names:
#@markdown >
#@markdown > `LEECHBOT_API_ID` · `LEECHBOT_API_HASH` · `LEECHBOT_BOT_TOKEN` · `LEECHBOT_USER_ID` · `LEECHBOT_DUMP_ID`

API_ID = 0 # @param {type:"integer"}
API_HASH = "" # @param {type:"string"}
BOT_TOKEN = "" # @param {type:"string"}
OWNER_ID = 0 # @param {type:"integer"}
DUMP_ID = 0 # @param {type:"integer"}

#@markdown ---
#@markdown ## ⚙️ Options
MOUNT_DRIVE = False # @param {type:"boolean"}
USE_GPU = True # @param {type:"boolean"}
AUTO_RESTART = True # @param {type:"boolean"}
REPO_BRANCH = "main" # @param ["main"]

# ═══════════════════════════════════════════════════════════
# 🚀 Deployment Engine — No edits needed below
# ═══════════════════════════════════════════════════════════

import subprocess, sys, os, json, time, shutil, signal
from pathlib import Path
from IPython.display import clear_output, display, Markdown
import logging

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["ALSA_CONFIG_PATH"] = "/dev/null"

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger("LeechBot")

# ─── UI Helpers ───────────────────────────────────────────
STEP = [0]

def banner():
    return """
    ╔═══════════════════════════════════════════╗
    ║         🚀 L E E C H B O T               ║
    ║    Advanced Telegram File Transloader     ║
    ╠═══════════════════════════════════════════╣
    ║  👤 Shinei Nouzen  ·  📂 Shineii86       ║
    ╚═══════════════════════════════════════════╝
    """

def log(emoji, msg, color="#2196F3"):
    display(Markdown(f'<font color="{color}">**{emoji} {msg}**</font>'))

def step(msg):
    STEP[0] += 1
    display(Markdown(f"\n---\n### Step {STEP[0]}: {msg}"))

def ok(msg):   log("✅", msg, "#4CAF50")
def fail(msg): log("❌", msg, "#F44336")
def warn(msg): log("⚠️", msg, "#FF9800")
def info(msg): log("ℹ️", msg, "#2196F3")

def run(cmd, desc, retries=3):
    for i in range(retries):
        try:
            info(f"{desc} (attempt {i+1}/{retries})")
            r = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True, timeout=300)
            ok(f"{desc} — done")
            return True
        except subprocess.CalledProcessError as e:
            if i == retries - 1:
                fail(f"{desc} failed: {e.stderr[:200]}")
                return False
            time.sleep(2 ** i)
        except subprocess.TimeoutExpired:
            if i == retries - 1:
                fail(f"{desc} timed out")
                return False
    return False

# ─── Credentials ──────────────────────────────────────────
def load_credentials():
    creds = {}
    try:
        from google.colab import userdata
        secrets = {
            'API_ID': 'LEECHBOT_API_ID', 'API_HASH': 'LEECHBOT_API_HASH',
            'BOT_TOKEN': 'LEECHBOT_BOT_TOKEN', 'OWNER_ID': 'LEECHBOT_USER_ID',
            'DUMP_ID': 'LEECHBOT_DUMP_ID'
        }
        for key, name in secrets.items():
            try:
                val = userdata.get(name)
                creds[key] = int(val) if key in ['API_ID', 'OWNER_ID', 'DUMP_ID'] else val
                ok(f"{key} loaded from Colab Secrets")
            except:
                creds[key] = None
    except ImportError:
        pass

    fallbacks = {'API_ID': API_ID, 'API_HASH': API_HASH, 'BOT_TOKEN': BOT_TOKEN,
                 'OWNER_ID': OWNER_ID, 'DUMP_ID': DUMP_ID}
    for k in fallbacks:
        if not creds.get(k):
            creds[k] = fallbacks[k]
    return creds

def validate(creds):
    required = ['API_ID', 'API_HASH', 'BOT_TOKEN', 'OWNER_ID', 'DUMP_ID']
    missing = [k for k in required if not creds.get(k)]
    if missing:
        fail(f"Missing: {', '.join(missing)}")
        return False
    # Auto-format DUMP_ID
    d = str(creds['DUMP_ID'])
    if len(d) == 10 and not d.startswith('-100'):
        creds['DUMP_ID'] = int(f"-100{d}")
        info("Auto-formatted DUMP_ID with -100 prefix")
    return True

# ─── Deploy ───────────────────────────────────────────────
def deploy():
    clear_output(wait=True)
    print(banner())

    # Step 1: Credentials
    step("🔐 Load Credentials")
    creds = load_credentials()
    if not validate(creds):
        fail("Fix credentials and re-run")
        return

    # Step 2: Clone
    step("📦 Clone Repository")
    os.chdir("/content")
    if os.path.exists("/content/leechbot"):
        shutil.rmtree("/content/leechbot")
        info("Cleaned previous install")
    if not run(f"git clone -b {REPO_BRANCH} --depth 1 https://github.com/Shineii86/LeechBot.git /content/leechbot",
               "Cloning LeechBot"):
        return

    # Step 3: Dependencies
    step("📦 Install Dependencies")
    if not run("apt-get update -qq && apt-get install -y -qq ffmpeg aria2 megatools p7zip-full unzip",
               "System packages"):
        return
    if not run("pip3 install -q --no-cache-dir -r /content/leechbot/requirements.txt",
               "Python packages"):
        return

    # Step 4: GPU Check
    step("🎮 Hardware Check")
    if USE_GPU:
        try:
            r = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                              shell=True, capture_output=True, text=True, check=True)
            name, mem = r.stdout.strip().split(', ')
            ok(f"GPU: {name} ({mem} VRAM)")
        except:
            info("No GPU — using CPU")
    else:
        info("GPU disabled by user")

    # Step 5: Save config
    step("💾 Save Configuration")
    cfg_path = "/content/leechbot/credentials.json"
    with open(cfg_path, 'w') as f:
        json.dump(creds, f, indent=2)
    os.chmod(cfg_path, 0o600)
    ok("credentials.json saved")

    # Set env vars
    os.environ["API_ID"] = str(creds["API_ID"])
    os.environ["API_HASH"] = str(creds["API_HASH"])
    os.environ["BOT_TOKEN"] = str(creds["BOT_TOKEN"])
    os.environ["OWNER_ID"] = str(creds["OWNER_ID"])
    os.environ["DUMP_ID"] = str(creds["DUMP_ID"])
    ok("Environment variables set")

    # Step 6: Mount Drive (optional)
    if MOUNT_DRIVE:
        step("☁️ Mount Google Drive")
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            ok("Drive mounted")
        except Exception as e:
            warn(f"Drive mount failed: {e}")

    # Step 7: Clean sessions
    step("🧹 Clean Sessions")
    for sf in ["/content/leechbot/leechbot_session.session",
               "/content/leechbot/leechbot_session.session-journal"]:
        if os.path.exists(sf):
            os.remove(sf)
            info(f"Removed: {sf}")
    ok("Sessions cleaned")

    # Step 8: Launch
    step("🚀 Launch Bot")
    clear_output(wait=True)
    print(banner())
    display(Markdown("""
### ✅ Deployment Complete!

| Command | Action |
|:--------|:-------|
| `/start` | Initialize bot |
| `/tupload` | Leech to Telegram |
| `/gdupload` | Mirror to Google Drive |
| `/ytupload` | YouTube / yt-dlp download |
| `/settings` | Bot preferences |
| `/cookies` | Check YouTube auth status |
| `/help` | All commands |

---
<sub>⚠️ Keep this tab open. Bot runs in this session.</sub>
"""))

    if AUTO_RESTART:
        signal.signal(signal.SIGTERM, lambda s, f: (info("Shutting down..."), sys.exit(0)))
        signal.signal(signal.SIGINT, lambda s, f: (info("Shutting down..."), sys.exit(0)))

    os.chdir("/content/leechbot")
    get_ipython().system('python3 -m leechbot')

# ─── Run ──────────────────────────────────────────────────
try:
    deploy()
except KeyboardInterrupt:
    warn("Cancelled by user")
except Exception as e:
    fail(f"Unexpected error: {e}")
    logger.exception("Full traceback:")

In [ ]:
# @title 🔍 Health Check
#@markdown Check if everything is set up correctly before deploying.

import os, subprocess, json
from IPython.display import display, Markdown

def check(label, ok, detail=""):
    icon = "✅" if ok else "❌"
    suffix = f" — `{detail}`" if detail else ""
    display(Markdown(f"{icon} **{label}**{suffix}"))

display(Markdown("## 🔍 Pre-Flight Check\n"))

# Python
v = sys.version.split()[0]
check("Python 3.10+", tuple(int(x) for x in v.split('.')) >= (3, 10), v)

# Node.js (needed for PO token plugin)
try:
    nj = subprocess.run("node --version", shell=True, capture_output=True, text=True).stdout.strip()
    check("Node.js 20+", tuple(int(x.strip('v')) for x in nj.split('.')) >= (20,), nj)
except:
    check("Node.js 20+", False, "not found")

# ffmpeg
try:
    subprocess.run("ffmpeg -version", shell=True, capture_output=True, check=True)
    check("ffmpeg", True)
except:
    check("ffmpeg", False, "install with: apt install ffmpeg")

# aria2
try:
    subprocess.run("aria2c --version", shell=True, capture_output=True, check=True)
    check("aria2c", True)
except:
    check("aria2c", False, "install with: apt install aria2")

# yt-dlp
try:
    yv = subprocess.run("yt-dlp --version", shell=True, capture_output=True, text=True).stdout.strip()
    check("yt-dlp", True, yv)
except:
    check("yt-dlp", False, "will be installed from requirements.txt")

# GPU
try:
    gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                          shell=True, capture_output=True, text=True, check=True).stdout.strip()
    check("GPU", True, gpu)
except:
    check("GPU", False, "CPU mode")

# Disk space
try:
    du = shutil.disk_usage("/content" if os.path.exists("/content") else "/")
    free_gb = du.free / (1024**3)
    check("Disk Space", free_gb > 5, f"{free_gb:.1f} GB free")
except:
    pass

# Credentials
display(Markdown("\n### 🔐 Credentials"))
try:
    from google.colab import userdata
    for name in ['LEECHBOT_API_ID', 'LEECHBOT_API_HASH', 'LEECHBOT_BOT_TOKEN', 'LEECHBOT_USER_ID', 'LEECHBOT_DUMP_ID']:
        try:
            userdata.get(name)
            check(name, True, "set")
        except:
            check(name, False, "not in Secrets")
except ImportError:
    display(Markdown("ℹ️ Colab Secrets not available — fill credentials in Deployer cell"))

display(Markdown("\n---\n> Fix any ❌ above before running the Deployer cell."))